# 13 Expand report holdings to monthly

目标：向后展开包含报告月最多6个月，并在每个扩展月重新计算股数、市值和权重。

- 输入：12报告月持仓；调整因子；CRSP月度价格
- 输出：monthly_holdings_weights.parquet及陈旧/权重审计
- 主键：wficn+month+permno
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：12
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import os
import duckdb
import pyarrow.parquet as pq
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import output_path, upstream_path

def esc(path): return str(path).replace("'","''")
report_path=upstream_path(ROOT,RUN_MODE,"12_adjust_shares_and_build_report_holdings","data/processed/holdings/report_month_holdings.parquet")
limit="" if not SMOKE else " LIMIT 500000"
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")
con.execute(f"""CREATE TEMP VIEW report_input AS
 SELECT * FROM read_parquet('{esc(report_path)}')""")
con.execute("""CREATE TEMP TABLE report_vintages AS
 WITH distinct_vintages AS (
   SELECT wficn,date_trunc('month',rdate)::DATE source_month,max(rdate) source_rdate
   FROM report_input
   WHERE date_trunc('month',rdate)::DATE<=DATE '2024-12-01'
   GROUP BY wficn,date_trunc('month',rdate)::DATE
 )
 SELECT *,lead(source_month) OVER(PARTITION BY wficn ORDER BY source_month)
   next_report_month
 FROM distinct_vintages""")
con.execute("""CREATE TEMP VIEW reports AS
 SELECT r.*,v.source_rdate,v.source_month,v.next_report_month,
   least(
     (v.source_month+INTERVAL '5 months')::DATE,
     coalesce((v.next_report_month-INTERVAL '1 month')::DATE,DATE '2024-12-01'),
     DATE '2024-12-01'
   ) end_month
 FROM report_input r JOIN report_vintages v
   ON r.wficn=v.wficn AND r.rdate=v.source_rdate""")
con.execute(f"""CREATE TEMP TABLE fac AS
 SELECT try_cast(permno AS BIGINT) permno,
        date_trunc('month',try_cast(mthcaldt AS DATE))::DATE factor_month,
        first(try_cast(mthcumfacshr AS DOUBLE)) mthcumfacshr
 FROM (
   SELECT permno,mthcaldt,mthcumfacshr
   FROM read_csv_auto('{esc(PATHS["mth_adjust_factor"])}',header=true,all_varchar=true)
   {limit}
 ) GROUP BY 1,2""")
con.execute(f"""CREATE TEMP TABLE price AS
 SELECT try_cast(PERMNO AS BIGINT) permno,
        date_trunc('month',try_cast(MthCalDt AS DATE))::DATE price_month,
        first(abs(try_cast(MthPrc AS DOUBLE))) monthly_price
 FROM (
   SELECT PERMNO,MthCalDt,MthPrc
   FROM read_csv_auto('{esc(PATHS["crsp_monthly"])}',header=true,all_varchar=true)
   {limit}
 ) GROUP BY 1,2""")
con.execute("""CREATE TEMP VIEW expanded AS
 SELECT r.*,
   (r.source_month+i*INTERVAL '1 month')::DATE month_value,
   i::INTEGER holding_age_months
 FROM reports r CROSS JOIN range(0,6) ages(i)
 WHERE (r.source_month+i*INTERVAL '1 month')::DATE<=r.end_month""")
con.execute("""CREATE TEMP VIEW monthly_joined AS
 SELECT e.* EXCLUDE(market_value,total_market_value,holding_weight),
   f.mthcumfacshr,p.monthly_price,
   CASE WHEN e.facshr_fdate>0 AND f.mthcumfacshr>0
     THEN e.shares*e.facshr_fdate/f.mthcumfacshr END shares_month
 FROM expanded e
 LEFT JOIN fac f ON e.permno=f.permno AND e.month_value=f.factor_month
 LEFT JOIN price p ON e.permno=p.permno AND e.month_value=p.price_month""")
con.execute("""CREATE TEMP VIEW monthly_valued AS
 SELECT *,
   CAST(shares_month AS DOUBLE)*CAST(monthly_price AS DOUBLE) market_value,
   facshr_fdate>0 AND mthcumfacshr>0 AND monthly_price>0
     AND shares_month IS NOT NULL valid_holding
 FROM monthly_joined""")
valued_stage=PATHS["full_run_tmp"]/"13_monthly_valued_stable.parquet"
valued_stage.unlink(missing_ok=True)
duckdb_copy_parquet_atomic(
    con,
    "SELECT * FROM monthly_valued ORDER BY wficn,month_value,permno",
    valued_stage,
)
con.execute(f"""CREATE TEMP VIEW stable_monthly_valued AS
 SELECT * FROM read_parquet('{esc(valued_stage)}')""")
con.execute("""CREATE TEMP TABLE canonical_month_totals AS
 SELECT wficn,month_value,
   fsum(CASE WHEN valid_holding THEN CAST(market_value AS DOUBLE) END)
     canonical_total_market_value
 FROM stable_monthly_valued
 GROUP BY wficn,month_value""")
con.execute("""CREATE TEMP VIEW monthly_result AS
 SELECT v.* EXCLUDE(month_value),v.month_value AS "month",
   t.canonical_total_market_value AS total_market_value,
   CASE WHEN v.valid_holding THEN
     CAST(v.market_value AS DOUBLE)
       /nullif(t.canonical_total_market_value,0) END holding_weight
 FROM stable_monthly_valued v
 JOIN canonical_month_totals t USING(wficn,month_value)""")
out=output_path(ROOT,RUN_MODE,"13_expand_report_holdings_to_monthly","data/processed/holdings/monthly_holdings_weights.parquet")
audit1=output_path(ROOT,RUN_MODE,"13_expand_report_holdings_to_monthly","data/audit/13_holding_staleness.csv")
audit2=output_path(ROOT,RUN_MODE,"13_expand_report_holdings_to_monthly","data/audit/13_monthly_weight_failures.csv")
audit3=output_path(ROOT,RUN_MODE,"13_expand_report_holdings_to_monthly","data/audit/13_monthly_weight_audit.csv")
staged=PATHS["full_run_tmp"]/"13_monthly_holdings_staged.parquet"
staged.unlink(missing_ok=True)
duckdb_copy_parquet_atomic(
    con,
    'SELECT * FROM monthly_result ORDER BY wficn,"month",permno',
    staged,
)
con.execute(f"""CREATE TEMP VIEW staged_monthly AS
 SELECT * FROM read_parquet('{esc(staged)}')""")
weight_audit_query=f"""SELECT
 wficn,"month",
 count(*) n_holdings_raw,
 count(*) FILTER(WHERE valid_holding) n_holdings_valid,
 fsum(CAST(market_value AS DOUBLE)) raw_market_value,
 max(total_market_value) valid_market_value,
 avg(CASE WHEN monthly_price>0 THEN 1.0 ELSE 0.0 END) price_coverage,
 avg(CASE WHEN facshr_fdate>0 AND mthcumfacshr>0 THEN 1.0 ELSE 0.0 END)
   adjustment_factor_coverage,
 fsum(CAST(holding_weight AS DOUBLE)) holding_weight_sum,
 abs(coalesce(fsum(CAST(holding_weight AS DOUBLE)),0)-1)
   weight_sum_deviation,
 CASE
   WHEN count(*) FILTER(WHERE valid_holding)=0 THEN 'no_valid_holdings'
   WHEN abs(fsum(CAST(holding_weight AS DOUBLE))-1)
     >{float(PARAMS["weight_sum_tolerance"])}
     THEN 'unexplained_weight_sum_deviation'
   WHEN avg(CASE WHEN monthly_price>0 THEN 1.0 ELSE 0.0 END)<1
     OR avg(CASE WHEN facshr_fdate>0 AND mthcumfacshr>0 THEN 1.0 ELSE 0.0 END)<1
     THEN 'excluded_holdings_missing_price_or_adjustment'
   ELSE 'complete'
 END reason_code
 FROM staged_monthly GROUP BY wficn,"month" """
weight_audit_stage=PATHS["full_run_tmp"]/"13_monthly_weight_audit_staged.parquet"
weight_audit_stage.unlink(missing_ok=True)
duckdb_copy_parquet_atomic(con,weight_audit_query,weight_audit_stage)
con.execute(f"""CREATE TEMP VIEW canonical_weight_audit AS
 SELECT * FROM read_parquet('{esc(weight_audit_stage)}')""")
duckdb_copy_csv_atomic(con,"""SELECT DISTINCT
 wficn,"month",source_rdate,source_month,next_report_month,end_month,holding_age_months
 FROM staged_monthly""",audit1)
duckdb_copy_csv_atomic(con,"""SELECT * FROM canonical_weight_audit
 WHERE reason_code IN ('no_valid_holdings','unexplained_weight_sum_deviation')""",audit2)
duckdb_copy_csv_atomic(con,"SELECT * FROM canonical_weight_audit",audit3)
report_count=int(pq.ParquetFile(report_path).metadata.num_rows)
monthly_count=int(pq.ParquetFile(staged).metadata.num_rows)
integrity=con.execute("""WITH grouped AS (
 SELECT wficn,"month",
   count(*) n_rows,
   count(DISTINCT permno) n_permnos,
   count(*) FILTER(WHERE valid_holding IS NOT TRUE) n_invalid,
   count(*) FILTER(WHERE source_month>"month") n_future,
   count(DISTINCT source_rdate) n_source_rdates,
   min(holding_age_months) min_age,
   max(holding_age_months) max_age
 FROM staged_monthly
 GROUP BY wficn,"month"
)
SELECT
 coalesce(sum(n_rows-n_permnos),0) duplicate_count,
 coalesce(sum(n_invalid),0) invalid_count,
 coalesce(sum(n_future),0) future_count,
 coalesce(sum(CASE WHEN n_source_rdates>1 THEN 1 ELSE 0 END),0)
   multi_vintage_count,
 min(min_age),max(max_age)
FROM grouped""").fetchone()
duplicate_count=int(integrity[0]); invalid_count=int(integrity[1])
future_count=int(integrity[2]); multi_vintage_count=int(integrity[3])
age_bounds=(integrity[4],integrity[5])
assert duplicate_count==0
assert age_bounds[0]>=0 and age_bounds[1]<=5
assert future_count==0 and multi_vintage_count==0
con.close()
out.parent.mkdir(parents=True,exist_ok=True)
os.replace(staged,out)
valued_stage.unlink(missing_ok=True)
weight_audit_stage.unlink(missing_ok=True)
INPUTS_USED=["report_month_holdings.parquet","mthadjustfactors.csv","monthly_stock_file.csv"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,audit1,audit2]]
OUTPUTS_WRITTEN.append(str(audit3.relative_to(ROOT)))
INPUT_COUNTS={"report_rows":report_count}; OUTPUT_COUNTS={"monthly_holdings":monthly_count}; PRIMARY_KEY=["wficn","month","permno"]
DUPLICATE_KEY_COUNT=duplicate_count; EXCLUDED_ROW_COUNT=invalid_count; EXCEPTION_FILE=str(audit2.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["Each extension month rejoined factor and price; maximum age is five months after the report month.","Monthly market value is recomputed in float64 and one canonical stable-order compensated total is reused for every holding weight.","The critical reduction is single-threaded; output is stably ordered by wficn, month, permno and is never rounded.","The pre-normalization coverage and post-normalization weight evidence were preserved in 13_monthly_weight_audit.csv."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "13_expand_report_holdings_to_monthly", stage="13", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
